# Phase 1 (Exploratory): Classical SVM benchmark on PCA-reduced breast cancer dataset

This notebook demonstrates how PCA dimensionality reduction affects the performance of a classical SVM baseline on the breast cancer dataset.


We import scikit‑learn modules for SVM, scaling, PCA, and evaluation.

In [1]:
import sys, os, time
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Ensure our notebook can import custom utilities from src/utils

In [2]:
import os, sys

# Notebook is in /app/src/phase1 → go up one level to /app/src
SRC_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

Custom utilities for logging results, plotting decision boundaries, and loading datasets.

In [3]:
from utils.logger import log_results
from utils.visualizer import plot_projected_decision_boundary
from utils.data_loader import load_dataset_from_config

Load dataset via config, separate features (X) and target (y).

In [4]:
df, cfg = load_dataset_from_config()

X = df.drop(columns=["target"]).values
y = df["target"].values

Split into train/test sets and scale features for SVM stability.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Reduce dimensionality to 4 principal components to explore performance trade‑offs.

In [6]:
pca = PCA(n_components=4)  # reduce to 4 principal components
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

Train a linear SVM on PCA‑reduced features.

In [7]:
clf = SVC(kernel="linear")

start = time.time()
clf.fit(X_train, y_train)
training_time = round(time.time() - start, 4)

Evaluate accuracy, generalization gap, and log results to CSV.

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
generalization_gap = round(train_accuracy - test_accuracy, 4)

metrics = {
    "model": "SVM_BreastCancer_PCA",
    "dataset": cfg["dataset"],
    "accuracy": test_accuracy,
    "train_accuracy": train_accuracy,
    "generalization_gap": generalization_gap,
    "training_time": training_time
}
log_results(metrics)

print("\n=== SVM Breast Cancer PCA Results ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

Visualize the decision boundary in PCA‑projected space.

In [ ]:
plot_projected_decision_boundary(clf, X_test, y_test, title="SVM Breast Cancer (PCA Projection)")